In [1]:
class Department:
    """Represents an academic department."""
    def __init__(self, name, code):
        self.name = name
        self.code = code
        self.courses = []

    def add_course(self, course):
        """Adds a course to the department's catalog."""
        if course not in self.courses:
            self.courses.append(course)
            print(f"Course '{course.title}' added to Department '{self.name}'.")

    def __str__(self):
        return f"{self.name} ({self.code})"


class Course:
    """Represents a specific course offered by a department."""
    def __init__(self, title, code, department):
        self.title = title
        self.code = code
        self.department = department
        self.students = []
        
        # Automatically register this course with the department upon creation
        department.add_course(self)

    def add_student(self, student):
        """Registers a student in the course roster."""
        if student not in self.students:
            self.students.append(student)
            print(f"-> Registered {student.name} in course roster for {self.code}.")

    def __str__(self):
        return f"{self.code}: {self.title}"


class Student:
    """Represents a university student."""
    def __init__(self, name, student_id):
        self.name = name
        self.student_id = student_id
        self.enrolled_courses = []

    def enroll(self, course):
        """Enrolls the student in a specific course."""
        if course in self.enrolled_courses:
            print(f"{self.name} is already enrolled in {course.title}.")
            return

        # 1. Add course to student's list
        self.enrolled_courses.append(course)
        
        # 2. Add student to course's roster (two-way linkage)
        course.add_student(self)
        
        print(f"SUCCESS: {self.name} enrolled in {course.title} ({course.department.name}).")

    def show_schedule(self):
        """Prints the student's current enrollment."""
        print(f"\n--- Schedule for {self.name} ---")
        if not self.enrolled_courses:
            print("No courses enrolled.")
        for course in self.enrolled_courses:
            print(f"{course.code}: {course.title} [{course.department.name}]")
        print("------------------------------\n")


# --- Driver Code to Demonstrate Functionality ---
if __name__ == "__main__":
    # 1. Create Departments
    cs_dept = Department("Computer Science", "CS")
    math_dept = Department("Mathematics", "MATH")

    # 2. Create Courses (They auto-link to their departments)
    cs101 = Course("Intro to Python", "CS101", cs_dept)
    cs102 = Course("Data Structures", "CS102", cs_dept)
    math201 = Course("Calculus I", "MATH201", math_dept)

    # 3. Create Students
    alice = Student("Alice Smith", "S001")
    bob = Student("Bob Jones", "S002")

    print("\n--- Processing Enrollments ---")
    # 4. Enroll Students
    # Enroll Alice in a CS course
    alice.enroll(cs101)
    
    # Enroll Alice in a Math course
    alice.enroll(math201)

    # Enroll Bob in a CS course
    bob.enroll(cs102)

    # 5. Show Results
    alice.show_schedule()
    bob.show_schedule()

    # Verify from the Department/Course side
    print(f"Students in {cs101.title}: {[s.name for s in cs101.students]}")

Course 'Intro to Python' added to Department 'Computer Science'.
Course 'Data Structures' added to Department 'Computer Science'.
Course 'Calculus I' added to Department 'Mathematics'.

--- Processing Enrollments ---
-> Registered Alice Smith in course roster for CS101.
SUCCESS: Alice Smith enrolled in Intro to Python (Computer Science).
-> Registered Alice Smith in course roster for MATH201.
SUCCESS: Alice Smith enrolled in Calculus I (Mathematics).
-> Registered Bob Jones in course roster for CS102.
SUCCESS: Bob Jones enrolled in Data Structures (Computer Science).

--- Schedule for Alice Smith ---
CS101: Intro to Python [Computer Science]
MATH201: Calculus I [Mathematics]
------------------------------


--- Schedule for Bob Jones ---
CS102: Data Structures [Computer Science]
------------------------------

Students in Intro to Python: ['Alice Smith']


In [2]:
from abc import ABC, abstractmethod

class Payment(ABC):
    """Abstract base class representing a payment method."""
    
    @abstractmethod
    def pay(self, amount):
        pass


class CashPayment(Payment):
    """Class for paying by cash."""
    
    def pay(self, amount):
        print(f"Processing cash payment of ${amount:.2f}...")
        print("Payment successful. Receipt printed.")


class ChequePayment(Payment):
    """Class for paying by cheque."""
    
    def __init__(self, cheque_number, bank_name):
        self.cheque_number = cheque_number
        self.bank_name = bank_name

    def pay(self, amount):
        print(f"Processing cheque payment of ${amount:.2f}...")
        print(f"Verifying Cheque #{self.cheque_number} from {self.bank_name}...")
        print("Payment authorized.")


class Bill:
    """Represents a customer's bill."""
    
    def __init__(self, items, total_amount):
        self.items = items
        self.total_amount = total_amount
        self.is_paid = False

    def settle_bill(self, payment_method):
        """
        Accepts a Payment object (CashPayment or ChequePayment) 
        to settle the bill.
        """
        if self.is_paid:
            print("This bill has already been paid.")
            return

        print(f"\n--- Settling Bill for {self.items} ---")
        print(f"Total Due: ${self.total_amount:.2f}")
        
        # Polymorphic call: The specific 'pay' method is called 
        # depending on whether payment_method is Cash or Cheque.
        payment_method.pay(self.total_amount)
        
        self.is_paid = True
        print("---------------------------------------")


# --- Driver Code ---
if __name__ == "__main__":
    # 1. Create a bill for a restaurant meal
    restaurant_bill = Bill("Dinner for Two", 85.50)

    # 2. Pay by Cash
    cash = CashPayment()
    restaurant_bill.settle_bill(cash)

    # 3. Create a bill for a utility service
    electric_bill = Bill("August Electricity", 150.00)

    # 4. Pay by Cheque
    cheque = ChequePayment(cheque_number="CHQ-998877", bank_name="National Bank")
    electric_bill.settle_bill(cheque)


--- Settling Bill for Dinner for Two ---
Total Due: $85.50
Processing cash payment of $85.50...
Payment successful. Receipt printed.
---------------------------------------

--- Settling Bill for August Electricity ---
Total Due: $150.00
Processing cheque payment of $150.00...
Verifying Cheque #CHQ-998877 from National Bank...
Payment authorized.
---------------------------------------


In [3]:
class Distance:
    """Represents a distance in Feet and Inches."""
    
    def __init__(self, feet, inches):
        self.feet = feet
        self.inches = inches
        self._normalize()

    def _normalize(self):
        """Helper to convert excess inches to feet."""
        if self.inches >= 12:
            extra_feet = self.inches // 12
            self.feet += extra_feet
            self.inches = self.inches % 12

    def __isub__(self, other):
        """
        Overloads the '-=' operator.
        Usage: d1 -= d2
        """
        if isinstance(other, Distance):
            # Subtract values
            self.feet -= other.feet
            self.inches -= other.inches
            
            # Handle negative inches by borrowing from feet
            while self.inches < 0:
                self.inches += 12
                self.feet -= 1
            
            return self  # __isub__ must return the modified object
        else:
            return NotImplemented

    def __str__(self):
        return f"{self.feet} ft {self.inches} in"

# --- Driver Code ---
if __name__ == "__main__":
    # Create two distance objects
    d1 = Distance(10, 2)  # 10 ft 2 in
    d2 = Distance(4, 6)   # 4 ft 6 in

    print(f"Original d1: {d1}")
    print(f"Original d2: {d2}")

    print("\nExecuting: d1 -= d2")
    # This calls d1.__isub__(d2)
    d1 -= d2

    print(f"New d1:      {d1}")
    
    # Verify d2 is unchanged
    print(f"d2 (unchanged): {d2}")

Original d1: 10 ft 2 in
Original d2: 4 ft 6 in

Executing: d1 -= d2
New d1:      5 ft 8 in
d2 (unchanged): 4 ft 6 in


In [4]:
class DistanceKM:
    """Represents distance in Kilometers."""
    
    def __init__(self, data):
        """
        Constructor that accepts either a raw number (kilometers)
        OR a DistanceMeters object to convert from.
        """
        # Check if the input is an instance of DistanceMeters
        # We use strict type checking or duck typing. 
        # Here we check if it has the 'meters' attribute.
        if hasattr(data, 'meters'):
            # Convert meters to kilometers
            self.kms = data.meters / 1000.0
            print(f"-> Converting {data.meters} meters to Kilometers...")
        else:
            # Assume it is a raw number
            self.kms = float(data)

    def display(self):
        print(f"Distance: {self.kms:.3f} km")


class DistanceMeters:
    """Represents distance in Meters."""
    
    def __init__(self, data):
        """
        Constructor that accepts either a raw number (meters)
        OR a DistanceKM object to convert from.
        """
        # Check if the input is an instance of DistanceKM
        if hasattr(data, 'kms'):
            # Convert kilometers to meters
            self.meters = data.kms * 1000.0
            print(f"-> Converting {data.kms} km to Meters...")
        else:
            # Assume it is a raw number
            self.meters = float(data)

    def display(self):
        print(f"Distance: {self.meters:.2f} m")


# --- Driver Code ---
if __name__ == "__main__":
    print("--- Creating Objects with Raw Values ---")
    # 1. Create a distance in Meters directly
    dm1 = DistanceMeters(500)  # 500 meters
    dm1.display()

    # 2. Create a distance in Kilometers directly
    dk1 = DistanceKM(2.5)      # 2.5 km
    dk1.display()

    print("\n--- converting Objects ---")
    
    # 3. Convert Meters to Kilometers
    # We pass the DistanceMeters object (dm1) into the DistanceKM constructor
    dk2 = DistanceKM(dm1)
    dk2.display()

    # 4. Convert Kilometers to Meters
    # We pass the DistanceKM object (dk1) into the DistanceMeters constructor
    dm2 = DistanceMeters(dk1)
    dm2.display()

--- Creating Objects with Raw Values ---
Distance: 500.00 m
Distance: 2.500 km

--- converting Objects ---
-> Converting 500.0 meters to Kilometers...
Distance: 0.500 km
-> Converting 2.5 km to Meters...
Distance: 2500.00 m
